# stVCR PFC dataset training

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Fit the three-age spatial model

In [ ]:
# ================================================================
# QIAN PFC — FINAL stVCR PUBLICATION RUN
# ONE STANDALONE JUPYTER CELL
# 3 tissues: GW15 F1b, GW20 F1, GW22 F2
# ================================================================

from pathlib import Path
import sys, gc, json, random, hashlib, platform, time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

# -------------------------
# 1. Configuration
# -------------------------

SMOKE = False  # True = quick test; False = full publication run

DATA_ROOT = Path(
    os.environ["QIAN_PFC_DATA_ROOT"]
)

PROJECT = Path(
    os.environ["STP_PROJECT"]
)

OUT = PROJECT / (
    "experiments/qian_pfc_publication_4way_20261024/"
    "stvcr_slices_1_2_7_official"
)

if SMOKE:
    OUT = OUT.with_name(OUT.name + "_SMOKE")

SEED = 19491001
EPOCHS = 3 if SMOKE else 2000
AE_EPOCHS = 4 if SMOKE else 1000
NUM_SAMPLES = 64 if SMOKE else 500
OT_DOWNSAMPLE = 256 if SMOKE else 5000

OUT.mkdir(parents=True, exist_ok=True)

torch.set_num_threads(8)
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

if not SMOKE and DEVICE.type != "cuda":
    raise RuntimeError(
        "Full stVCR training requires a GPU-backed Jupyter kernel. "
        "Use SMOKE=True to test on CPU."
    )

print("Device:", DEVICE)
print("Publication run:", not SMOKE)
print("Output:", OUT)

# -------------------------
# 2. Imports
# -------------------------

sys.path.insert(0, str(PROJECT / "src"))

for repo in [
    PROJECT / "stVCR/src",
    PROJECT / "stvcr/src",
    PROJECT / "stVCR",
    PROJECT / "stvcr",
]:
    if (repo / "stvcr" / "__init__.py").exists():
        sys.path.insert(0, str(repo))

import scanpy as sc
import stvcr as st

from stvcr.training.train import default_config

from flow_stp.data.qian_pfc import (
    QianPFCConfig,
    build_qian_pfc_panel,
    load_qian_pfc_for_stage1,
)

# -------------------------
# 3. Frozen section selection
# -------------------------

cfg = QianPFCConfig(
    data_root=DATA_ROOT,
    targets=(
        ("GW15", ("F1a", "F1b")),
        ("GW20", ("F1", "F2a", "F2b")),
        ("GW22", ("F1", "F2", "F3")),
    ),
    expected_num_slices=9,
    max_cells_per_slice=2000 if SMOKE else None,
    layer_key_for_counts="raw",
)

panel9 = build_qian_pfc_panel(cfg)

expected = [
    ("GW15", "UMB1117", "F1a"),
    ("GW15", "UMB1117", "F1b"),
    ("GW20", "FB080", "F1"),
    ("GW20", "FB080", "F2a"),
    ("GW20", "FB080", "F2b"),
    ("GW20", "FB121", "F1"),
    ("GW22", "FB123", "F1"),
    ("GW22", "FB123", "F2"),
    ("GW22", "FB123", "F3"),
]

actual = [
    (str(r.gw), str(r["sample"]), str(r.region))
    for _, r in panel9.iterrows()
]

assert actual == expected, (
    "Nine-section inventory does not match the frozen benchmark.",
    actual,
)

selected_ids = [1, 2, 7]

panel3 = panel9[
    panel9.slice_id.isin(selected_ids)
].copy()

panel3["original_slice_id"] = panel3["slice_id"]
panel3["slice_id"] = np.arange(3)

panel9.to_csv(OUT / "nine_tissue_inventory.csv", index=False)
panel3.to_csv(OUT / "selected_tissues.csv", index=False)

print("\nSelected tissues:")
print(panel3[[
    "gw", "sample", "region",
    "original_slice_id", "n_full"
]].to_string(index=False))

# -------------------------
# 4. Load or preprocess
# -------------------------

PREP = OUT / "stvcr_preprocessed_3tissue.h5ad"
MODEL = OUT / "stvcr_model_full.pt"
RIGID = OUT / "stvcr_rigid_transform.pt"

if (OUT / "SUCCESS.json").exists():
    raise FileExistsError(
        f"Completed run already exists: {OUT}"
    )

if MODEL.exists() or RIGID.exists():
    raise FileExistsError(
        "Existing stVCR checkpoint detected. "
        "Use a new output directory to avoid overwriting."
    )

if PREP.exists():

    print("\nReusing preprocessing:", PREP)
    ad = sc.read_h5ad(PREP)

else:

    qian = load_qian_pfc_for_stage1(
        cfg,
        panel_df=panel3,
    )

    counts_per_slice = tuple(
        len(x) for x in qian.counts_by_slice
    )

    print("\nLoaded cells:", counts_per_slice)
    print("Genes:", qian.num_genes)

    if not SMOKE:
        assert counts_per_slice == (
            410492, 665204, 439986
        ), "Unexpected cell counts"

        assert qian.num_genes == 300

    matrices = []
    coords = []
    obs_parts = []

    for k in range(3):

        Y = qian.counts_by_slice[k].numpy()

        assert np.isfinite(Y).all()
        assert np.min(Y) >= 0

        matrices.append(sp.csr_matrix(Y))

        coords.append(
            np.asarray(
                qian.spatial_by_slice[k],
                dtype=np.float32
            )
        )

        frame = qian.obs_df[
            qian.obs_df.slice_id.eq(k)
        ].copy()

        frame["original_slice_id"] = selected_ids[k]

        obs_parts.append(frame)

    genes = list(qian.gene_names)

    ad = sc.AnnData(
        X=sp.vstack(matrices, format="csr"),
        obs=pd.concat(obs_parts, ignore_index=True),
        var=pd.DataFrame(
            index=pd.Index(genes, name="gene")
        ),
    )

    del qian, matrices, Y
    gc.collect()

    ad.obs_names = pd.Index(
        [
            f"{i}|{cell}"
            for i, cell in zip(
                ad.obs.original_slice_id,
                ad.obs.cell_id
            )
        ],
        name="cell_key",
    )

    assert ad.obs_names.is_unique

    ad.obsm["spatial"] = np.vstack(coords)

    # GW15 -> 0, GW20 -> 5/7, GW22 -> 1
    ad.obs["time"] = (
        pd.to_numeric(ad.obs["time"]) - 15.0
    ) / 7.0

    assert np.allclose(
        np.sort(ad.obs.time.unique()),
        [0.0, 5.0 / 7.0, 1.0]
    )

    ad.obs[
        ["gw", "sample", "region",
         "original_slice_id", "time"]
    ].to_csv(
        OUT / "selected_cells.csv.gz"
    )

    # Library-normalized log expression for official AE
    # No gene or cell filtering.
    sc.pp.normalize_total(ad, target_sum=1e4)
    sc.pp.log1p(ad)

    # Compatibility with newer scipy versions
    if not hasattr(sp.csr_matrix, "A"):
        sp.csr_matrix.A = property(
            lambda self: self.toarray()
        )

    print("\nTraining stVCR autoencoder...", flush=True)

    st.pp.pp_stvcr(
        ad,
        use_pp_with_scanpy=False,
        ae_model_save_path=str(OUT / "stvcr_ae.pt"),
        z_dims=10,
        device=DEVICE,
        n_epochs=AE_EPOCHS,
        batch_size=1000,
        early_stop=30,
        seed=SEED,
        spatial_key="spatial",
        time_key="time",
        use_initial_alignment=True,
        alpha=0.002,
        down_sampling_number=OT_DOWNSAMPLE,
    )

    assert ad.obsm["X_gene_input"].shape == (
        ad.n_obs, 10
    )

    assert ad.obsm["X_spatial_input"].shape == (
        ad.n_obs, 2
    )

    assert np.isfinite(ad.obsm["X_gene_input"]).all()
    assert np.isfinite(ad.obsm["X_spatial_input"]).all()

    print("\nSaving preprocessed dataset...", flush=True)

    ad.write_h5ad(PREP, compression="gzip")
    gc.collect()

# -------------------------
# 5. Official stVCR training
# -------------------------

config = dict(default_config)

config["n_epochs"] = EPOCHS
config["num_samples"] = NUM_SAMPLES

manifest = {
    "method": "Official stVCR",
    "dataset": "Qian PFC",
    "original_slice_ids": selected_ids,
    "cell_count": int(ad.n_obs),
    "gene_count": int(ad.n_vars),
    "smoke": SMOKE,
    "seed": SEED,
    "device": str(DEVICE),
    "autoencoder_epochs": AE_EPOCHS,
    "training_config": config,
    "preprocessing": (
        "Raw 300-gene counts; library normalize 1e4; "
        "log1p; official 10D AE; official OT alignment"
    ),
    "data_root": str(DATA_ROOT),
    "python": sys.version,
    "torch": torch.__version__,
    "host": platform.node(),
}

(OUT / "run_manifest.json").write_text(
    json.dumps(manifest, indent=2, default=str)
)

(OUT / "training_config.json").write_text(
    json.dumps(config, indent=2)
)

print("\n" + "=" * 60)
print("STARTING FINAL stVCR TRAINING")
print("=" * 60)
print("Cells:", ad.n_obs)
print("Genes:", ad.n_vars)
print("Timepoints:", sorted(ad.obs.time.unique()))
print("Epochs:", EPOCHS)
print("Samples:", NUM_SAMPLES)
print("Device:", DEVICE)
print("=" * 60, flush=True)

start = time.perf_counter()

st.tr.train_stvcr(
    ad,
    model_path=str(MODEL),
    rigid_transformation_path=str(RIGID),
    config=config,
    use_gene=True,
    use_spatial=True,
    use_growth=True,
    use_alignment=True,
    device=DEVICE,
)

assert MODEL.is_file()
assert RIGID.is_file()

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(2**22), b""):
            h.update(chunk)
    return h.hexdigest()

result = {
    "status": "success",
    "elapsed_seconds": time.perf_counter() - start,
    "model": str(MODEL),
    "rigid_transformation": str(RIGID),
    "model_sha256": sha256(MODEL),
    "rigid_sha256": sha256(RIGID),
    "preprocessed_sha256": sha256(PREP),
}

(OUT / "SUCCESS.json").write_text(
    json.dumps(result, indent=2)
)

print("\n" + "=" * 60)
print("stVCR TRAINING COMPLETE")
print("=" * 60)
print("Model:", MODEL)
print("Rigid alignment:", RIGID)
print("Output:", OUT)

## Fit the strict GW20 holdout model

In [ ]:
# ======================================================================
# QIAN PFC — stVCR STRICT GW20 HOLDOUT (ONE STANDALONE JUPYTER CELL)
# ======================================================================
# Fit stVCR from scratch using ONLY:
#   GW15 / UMB1117 / F1b: 410,492 cells (original 9-panel slice ID 1)
#   GW22 / FB123   / F2 : 439,986 cells (original 9-panel slice ID 7)
# NO GW20 tissue, counts, spatial coordinates, or labels are read.
# Exact same preprocessing/training protocol as the completed three-tissue
# PFC stVCR run; only the training-tissue selection and output folder change.
# GW20 prediction later uses normalized t = (20 - 15)/(22 - 15) = 5/7.
# ======================================================================

from pathlib import Path
import os, sys, gc, json, time, random, hashlib, inspect, subprocess, platform
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

# ------------------------- Configuration -----------------------------
SMOKE = False                 # True: 2,000 cells per training tissue
SEED = 19491001
DATA_ROOT = Path(
    os.environ["QIAN_PFC_DATA_ROOT"]
)
PROJECT = Path(
    os.environ["STP_PROJECT"]
)
OUT = (
    PROJECT / "experiments/qian_pfc_publication_4way_20261024/"
    "stvcr_gw15_gw22_all_gw20_heldout_official"
)
if SMOKE:
    OUT = OUT.with_name(OUT.name + "_SMOKE")
OUT.mkdir(parents=True, exist_ok=True)

EPOCHS = 3 if SMOKE else 2000
AE_EPOCHS = 4 if SMOKE else 1000
NUM_SAMPLES = 64 if SMOKE else 500
OT_DOWNSAMPLE = 256 if SMOKE else 5000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not SMOKE and DEVICE.type != "cuda":
    raise RuntimeError("Full fit requires a GPU-backed Jupyter kernel.")

NUM_THREADS = min(8, os.cpu_count() or 1)
torch.set_num_threads(NUM_THREADS)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PREP = OUT / "stvcr_preprocessed_gw15_gw22.h5ad"
AE = OUT / "stvcr_ae_gw15_gw22.pt"
MODEL = OUT / "stvcr_model_gw15_gw22.pt"
RIGID = OUT / "stvcr_rigid_gw15_gw22.pt"
SUCCESS = OUT / "SUCCESS.json"

if SUCCESS.exists():
    raise FileExistsError(f"A completed holdout fit already exists: {SUCCESS}")
if MODEL.exists() or RIGID.exists():
    raise FileExistsError(
        "Existing model/rigid checkpoint detected. Use a new OUT directory; "
        "do not silently overwrite a partially completed publication run."
    )

print("Device:", DEVICE)
print("SMOKE:", SMOKE, "| epochs:", EPOCHS)
print("STRICT holdout: GW20 will NOT be read or fitted")
print("Output:", OUT)

# ----------------------- Original repo imports -----------------------
sys.path.insert(0, str(PROJECT / "src"))
for source in [
    Path(str(Path(os.environ["STVCR_REPO"]) / "src")),
    PROJECT / "stVCR/src", PROJECT / "stvcr/src",
    PROJECT / "stVCR", PROJECT / "stvcr",
]:
    if (source / "stvcr" / "__init__.py").is_file():
        sys.path.insert(0, str(source))
        break

import scanpy as sc
import stvcr as st
from stvcr.training.train import default_config
from flow_stp.data.qian_pfc import (
    QianPFCConfig, build_qian_pfc_panel, load_qian_pfc_for_stage1,
)
print("stVCR package:", inspect.getfile(st))

# --------------- Select only two physical Qian sections ---------------
# Building this panel does not open GW20 h5ad files.
cfg = QianPFCConfig(
    data_root=DATA_ROOT,
    targets=(
        ("GW15", ("F1b",)),
        ("GW22", ("F2",)),
    ),
    expected_num_slices=2,
    max_cells_per_slice=2000 if SMOKE else None,
    layer_key_for_counts="raw",
)
panel = build_qian_pfc_panel(cfg)
assert len(panel) == 2
observed = [
    (str(r["gw"]), str(r["sample"]), str(r["region"]))
    for _, r in panel.iterrows()
]
assert observed == [
    ("GW15", "UMB1117", "F1b"),
    ("GW22", "FB123", "F2"),
], f"Wrong training sections: {observed}"
assert not panel["gw"].astype(str).str.contains("GW20").any()
panel["original_nine_slice_id"] = [1, 7]
print("\nFrozen training tissues (no GW20):")
print(panel[["gw", "sample", "region", "n_full", "original_nine_slice_id"]].to_string(index=False))
if not SMOKE:
    assert panel["n_full"].astype(int).tolist() == [410492, 439986]
panel.to_csv(OUT / "training_tissues_only.csv", index=False)

# --------------------- Load and preprocess ----------------------------
if PREP.exists():
    print("\nReusing existing GW15/GW22-only preprocessing:", PREP)
    ad = sc.read_h5ad(PREP)
    assert set(ad.obs["gw"].astype(str).unique()) == {"GW15", "GW22"}
    assert set(ad.obs["original_slice_id"].astype(int).unique()) == {1, 7}
    assert set(np.round(ad.obs["time"].astype(float), 7)) == {0., 1.}
    assert ad.n_vars == 300
else:
    qian = load_qian_pfc_for_stage1(cfg, panel_df=panel)
    section_counts = [len(x) for x in qian.counts_by_slice]
    expected_counts = [2000, 2000] if SMOKE else [410492, 439986]
    assert section_counts == expected_counts, (section_counts, expected_counts)
    assert qian.num_genes == 300
    assert set(qian.obs_df["gw"].astype(str)) == {"GW15", "GW22"}
    assert qian.obs_df["gw"].astype(str).str.contains("GW20").sum() == 0

    matrices, coords, obs_chunks = [], [], []
    for k, original_id in enumerate([1, 7]):
        Y = qian.counts_by_slice[k].numpy()
        assert Y.ndim == 2 and Y.shape[1] == 300
        assert np.isfinite(Y).all() and np.min(Y) >= 0
        matrices.append(sp.csr_matrix(Y))
        coords.append(np.asarray(qian.spatial_by_slice[k], dtype=np.float32))
        obs_k = qian.obs_df.loc[qian.obs_df["slice_id"].eq(k)].copy()
        obs_k["original_slice_id"] = original_id
        obs_chunks.append(obs_k)

    ad = sc.AnnData(
        X=sp.vstack(matrices, format="csr"),
        obs=pd.concat(obs_chunks, ignore_index=True),
        var=pd.DataFrame(index=pd.Index(qian.gene_names, name="gene")),
    )
    ad.obs_names = pd.Index(
        [f"{idx}|{cell}" for idx, cell in zip(ad.obs["original_slice_id"], ad.obs["cell_id"])],
        name="cell_key",
    )
    assert ad.obs_names.is_unique
    ad.obsm["spatial"] = np.vstack(coords)
    # Retain the original full three-age developmental clock.
    ad.obs["time"] = (pd.to_numeric(ad.obs["time"]) - 15.0) / 7.0
    assert np.allclose(np.sort(ad.obs["time"].unique()), [0., 1.])
    assert ad.n_obs == sum(expected_counts)

    # Exact same normalization as the completed full PFC stVCR run.
    ad.obs[["gw", "sample", "region", "original_slice_id", "time"]].to_csv(
        OUT / "training_cells_only.csv.gz"
    )
    sc.pp.normalize_total(ad, target_sum=1e4)
    sc.pp.log1p(ad)

    # Compatibility fix required by upstream AE implementation on new SciPy.
    if not hasattr(sp.csr_matrix, "A"):
        sp.csr_matrix.A = property(lambda self: self.toarray())

    print("\nTraining fresh 10D autoencoder on GW15 and GW22 ONLY...", flush=True)
    st.pp.pp_stvcr(
        ad,
        use_pp_with_scanpy=False,
        ae_model_save_path=str(AE),
        z_dims=10,
        device=DEVICE,
        n_epochs=AE_EPOCHS,
        batch_size=1000,
        early_stop=30,
        seed=SEED,
        spatial_key="spatial",
        time_key="time",
        use_initial_alignment=True,
        alpha=0.002,
        down_sampling_number=OT_DOWNSAMPLE,
    )
    assert ad.obsm["X_gene_input"].shape == (ad.n_obs, 10)
    assert ad.obsm["X_spatial_input"].shape == (ad.n_obs, 2)
    assert np.isfinite(ad.obsm["X_gene_input"]).all()
    assert np.isfinite(ad.obsm["X_spatial_input"]).all()
    assert set(ad.obs["gw"].astype(str)) == {"GW15", "GW22"}
    print("Saving GW15/GW22-only preprocessed data...", flush=True)
    ad.write_h5ad(PREP, compression="gzip")
    del qian, matrices, coords, obs_chunks, Y
    gc.collect()

# ------------------- Freeze training configuration --------------------
config = dict(default_config)
config["n_epochs"] = EPOCHS
config["num_samples"] = NUM_SAMPLES
assert np.isclose(config["kappa_exp"], 0.02)
assert np.isclose(config["alpha_exp"], 0.02)

# These checks rule out accidental reuse of three-tissue preprocessing.
assert set(ad.obs["gw"].astype(str).unique()) == {"GW15", "GW22"}
assert set(ad.obs["original_slice_id"].astype(int).unique()) == {1, 7}
assert not np.isclose(ad.obs["time"].astype(float), 5/7).any()
assert ad.n_obs == (4000 if SMOKE else 850478)
assert ad.n_vars == 300

def git_identity(src_file):
    p = Path(src_file).resolve()
    for base in [p.parent, *p.parents]:
        if (base / ".git").exists():
            try:
                sha = subprocess.check_output(
                    ["git", "-C", str(base), "rev-parse", "HEAD"],
                    text=True, stderr=subprocess.DEVNULL,
                ).strip()
                dirty = bool(subprocess.check_output(
                    ["git", "-C", str(base), "status", "--porcelain"],
                    text=True, stderr=subprocess.DEVNULL,
                ).strip())
                return {"commit": sha, "dirty_worktree": dirty, "repository": str(base)}
            except Exception:
                pass
    return {"source_file": str(p), "commit": None}

manifest = {
    "method": "stVCR official model and defaults",
    "purpose": "strict GW20 timepoint holdout for expression interpolation",
    "train_tissues": ["GW15 UMB1117 F1b", "GW22 FB123 F2"],
    "excluded_tissues": "ALL GW20 sections; no GW20 data loaded in training",
    "original_nine_slice_ids": [1, 7],
    "n_training_cells": int(ad.n_obs),
    "n_genes": int(ad.n_vars),
    "t_gw15": 0.0, "t_gw20_prediction": 5 / 7, "t_gw22": 1.0,
    "smoke": SMOKE, "seed": SEED, "device": str(DEVICE),
    "autoencoder_max_epochs": AE_EPOCHS,
    "preprocessing": "original raw 300-gene counts -> normalize_total(1e4) -> log1p -> official 10D AE; official initial OT rigid alignment",
    "training": "official st.tr.train_stvcr; use_gene/spatial/growth/alignment=True",
    "training_config": config,
    "stvcr_source": git_identity(inspect.getfile(st)),
    "input_files": [
        {"path": str(DATA_ROOT / name), "bytes": (DATA_ROOT / name).stat().st_size}
        for name in ("gw15.h5ad", "gw22.h5ad")
    ],
    "python": sys.version, "torch": torch.__version__, "host": platform.node(),
}
(OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
(OUT / "training_config.json").write_text(json.dumps(config, indent=2))

# ------------------------ Official fit -------------------------------
print("\n" + "=" * 72)
print("STARTING STRICT GW20 HOLDOUT stVCR TRAINING")
print("Training ages:", np.sort(ad.obs["time"].unique()).tolist())
print("GW20 evaluation age (NOT TRAINED):", 5 / 7)
print("Training cells:", f"{ad.n_obs:,}", "| Genes:", ad.n_vars)
print("Native fit epochs:", EPOCHS, "| Initial samples:", NUM_SAMPLES)
print("Config weights: kappa_exp=0.02, alpha_exp=0.02")
print("Model output:", MODEL)
print("=" * 72, flush=True)

start = time.perf_counter()
st.tr.train_stvcr(
    ad,
    model_path=str(MODEL),
    rigid_transformation_path=str(RIGID),
    config=config,
    use_gene=True,
    use_spatial=True,
    use_growth=True,
    use_alignment=True,
    device=DEVICE,
)
assert MODEL.is_file() and RIGID.is_file()

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fp:
        for chunk in iter(lambda: fp.read(8 * 1024**2), b""):
            h.update(chunk)
    return h.hexdigest()

finished = {
    "status": "success",
    "gw20_holdout": "all sections",
    "train_ages": [0.0, 1.0],
    "prediction_time_gw20": 5 / 7,
    "training_n_cells": int(ad.n_obs),
    "training_epochs": EPOCHS,
    "elapsed_seconds": time.perf_counter() - start,
    "model": str(MODEL), "rigid_transformation": str(RIGID),
    "model_sha256": sha256(MODEL),
    "rigid_sha256": sha256(RIGID),
    "preprocessed_sha256": sha256(PREP),
}
SUCCESS.write_text(json.dumps(finished, indent=2))
print("\nTRAINING COMPLETE — strict GW20 holdout, no GW20 training observations")
print("Saved:", OUT)
print("Model:", MODEL)
print("Manifest:", OUT / "run_manifest.json")
print("Future GW20 prediction time: t=5/7 (NO GW20 refit)")

## Export the strict holdout forecast

In [ ]:
# Qian PFC — stVCR prediction from the newest STRICT GW20-held-out checkpoint.
# PASTE AS ONE CELL into "stVCR PFC dataset training (2).ipynb".
# Run in its original "Python (stVCR benchmark)" kernel.
# READS ONLY GW15/GW22 preprocessing and frozen checkpoints; NEVER reads GW20.
# Does not retrain or modify any model. Preserves existing frozen forecasts.

from pathlib import Path
import json, hashlib, gc
import numpy as np
import torch
import scanpy as sc
# Load the unmodified authors' numerical evolution utility DIRECTLY.
# Importing stvcr.downstream.utils via the package executes
# downstream/__init__.py, which imports optional 3D visualization
# dependencies such as spateo (not needed for prediction).
import importlib.util

STVCR_UTILS = Path(
    str(Path(os.environ["STVCR_REPO"]) / "src/stvcr/downstream/utils.py")
)
if not STVCR_UTILS.is_file():
    raise FileNotFoundError(STVCR_UTILS)
_spec = importlib.util.spec_from_file_location(
    'stvcr_native_evolution_utils', str(STVCR_UTILS)
)
if _spec is None or _spec.loader is None:
    raise ImportError(f'Cannot load native stVCR utils from {STVCR_UTILS}')
_native = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_native)
evolution_forward_sim_rgb_data = _native.evolution_forward_sim_rgb_data
print('Loaded unmodified native stVCR evolution utility:', STVCR_UTILS)


# The exact locations in the completed strict-held-out training notebook.
PROJECT = Path(os.environ["STP_PROJECT"])
RUN_ROOT = PROJECT / 'experiments/qian_pfc_publication_4way_20261024'
TRAIN = RUN_ROOT / 'stvcr_gw15_gw22_all_gw20_heldout_official'
PRED_DIR = RUN_ROOT / 'GW20_heldout_fourway_Fisher_flow_vs_native_v1/frozen_predictions'
PRED_FILE = PRED_DIR / 'stvcr.npz'
GENES_FILE = (PROJECT / 'experiments/'
              'qian_pfc_gw20_allheldout_gw15_gw22_five_tissue_v1/'
              'CPU_FULL/gene_names.json')
SUCCESS = TRAIN / 'SUCCESS.json'
PREP = TRAIN / 'stvcr_preprocessed_gw15_gw22.h5ad'
MODEL = TRAIN / 'stvcr_model_gw15_gw22.pt'
AE = TRAIN / 'stvcr_ae_gw15_gw22.pt'

SEED = 20261027  # Identical to the four-method benchmark
N_PRED = 4096
GW20_TIME = 5.0 / 7.0

# Fail rather than silently writing over any already-frozen result.
if PRED_FILE.exists():
    raise FileExistsError(f'Existing prediction preserved; no overwrite: {PRED_FILE}')
for path in (SUCCESS, PREP, MODEL, AE, GENES_FILE):
    if not path.is_file():
        raise FileNotFoundError(f'Missing required training artifact: {path}')

run = json.loads(SUCCESS.read_text())
assert run.get('status') == 'success', 'Training did not finish successfully'
assert run.get('gw20_holdout') == 'all sections', 'This is NOT the strict GW20-held-out fit'
assert int(run.get('training_n_cells', -1)) == 850478, 'Unexpected training sample count'
assert np.isclose(float(run.get('prediction_time_gw20')), GW20_TIME)

# Native stVCR training used torch.save(model), not just a state_dict.
# Loading torch.save(model) in the training kernel preserves its model classes.
# Downstream visualization __init__ imports are avoided above.
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(4 * 1024**2), b''):
            h.update(chunk)
    return h.hexdigest()

assert sha256(MODEL) == run['model_sha256'], 'Trained model checksum mismatch'
expected_genes = [str(x) for x in json.loads(GENES_FILE.read_text())]
assert len(expected_genes) == 300 and len(set(expected_genes)) == 300

# 'backed=r' only reads the selected GW15 rows instead of the full expression matrix.
ad = sc.read_h5ad(PREP, backed='r')
try:
    assert ad.n_obs == 850478 and ad.n_vars == 300
    actual_genes = ad.var_names.astype(str).tolist()
    assert actual_genes == expected_genes, 'Gene order differs from Spatial TI (STOP: no silent reordering)'
    assert {'X_spatial_input', 'X_gene_input'} <= set(ad.obsm.keys())
    gw = ad.obs['gw'].astype(str).to_numpy()
    assert set(np.unique(gw)) == {'GW15', 'GW22'}
    assert np.sum(gw == 'GW15') == 410492
    assert np.sum(gw == 'GW22') == 439986
    assert set(ad.obs['original_slice_id'].astype(int).unique()) == {1, 7}
    assert np.allclose(np.sort(ad.obs['time'].astype(float).unique()), [0.0, 1.0])

    # Select source cells in sorted order (also compatible with backed HDF5).
    source_rows = np.flatnonzero(gw == 'GW15')
    pick = np.random.default_rng(SEED + 333).choice(
        source_rows, size=N_PRED, replace=False)
    pick.sort()
    spatial = np.asarray(ad.obsm['X_spatial_input'][pick], dtype=np.float32)
    latent = np.asarray(ad.obsm['X_gene_input'][pick], dtype=np.float32)
    initial_state = np.column_stack([spatial, latent]).astype(np.float32)
    assert initial_state.shape == (N_PRED, 12)
    assert np.isfinite(initial_state).all()
finally:
    ad.file.close()
    del ad
    gc.collect()

print('stVCR native prediction — newest strict GW20 holdout')
print('Training input: GW15 Tissue 2 + GW22 Tissue 2 only')
print('Starting GW15 cells:', N_PRED, '| target normalized time:', GW20_TIME)
print('Loading trained model and AE (NO TRAINING)...', flush=True)
model = torch.load(MODEL, map_location='cpu', weights_only=False)
ae = torch.load(AE, map_location='cpu', weights_only=False)
model.eval()
ae.eval()

# Exact official author evolution helper: evolves the joint spatial+gene
# state using learned velocities AND the stochastic growth mechanism.
np.random.seed(SEED + 334)
torch.manual_seed(SEED + 334)
with torch.no_grad():
    # Native stVCR returns five values:
# spatial states, expression states, cell types, cell IDs, times

    _, expression_series, _, _, native_times = evolution_forward_sim_rgb_data(
    torch.from_numpy(initial_state),
    model, 0.0, GW20_TIME,
    spatial_dim=2, delta_t=0.1,
)
    final_latent = np.asarray(expression_series[-1], dtype=np.float32)
    assert final_latent.ndim == 2 and final_latent.shape[1] == 10
    actual_time = float(native_times[-1])
    assert abs(actual_time - GW20_TIME) <= 0.01, (actual_time, GW20_TIME)
    assert len(final_latent) > 0, 'Native growth model produced no surviving cells'

    decoded_chunks = []
    for start in range(0, len(final_latent), 1024):
        z = torch.from_numpy(final_latent[start:start + 1024])
        decoded_chunks.append(ae.decode_mlp1(z).cpu().numpy())
    predicted_logcp10k = np.concatenate(decoded_chunks, axis=0)

del model, ae, expression_series

# Training input was normalize_total(target_sum=1e4) + log1p.
# Undo log1p to obtain positive expression intensity, then normalize
# rows for the common four-method gene-share comparison.
intensity = np.expm1(np.clip(predicted_logcp10k.astype(np.float64), -30., 30.))
negative_fraction = float(np.mean(intensity < 0))
intensity = np.maximum(intensity, 0.0)
assert np.isfinite(intensity).all() and (intensity.sum(axis=1) > 0).all()

# Same final sampling rule as the previous four-method benchmark adapter.
chosen = np.random.default_rng(SEED + 3333).choice(
    len(intensity), size=N_PRED, replace=len(intensity) < N_PRED)
chosen_intensity = intensity[chosen]
shares = (chosen_intensity / chosen_intensity.sum(axis=1, keepdims=True)).astype(np.float32)
assert shares.shape == (N_PRED, 300)
assert np.isfinite(shares).all() and np.allclose(shares.sum(1), 1., atol=1e-5)

metadata = {
    'method': 'stVCR',
    'source_t': 0.0,
    'target_t_normalized': GW20_TIME,
    'actual_native_time': actual_time,
    'training_sections': 2,
    'train_n_cells': 850478,
    'prediction_rule': 'native forward evolution (including growth), 10D AE decoded',
    'source_spatial_key': 'X_spatial_input (first timepoint unchanged by stVCR rigid alignment)',
    'decoder': 'inverse log1p normalized expression, gene shares',
    'negative_decoded_fraction': negative_fraction,
    'n_population_after_growth': int(len(intensity)),
    'checkpoint': str(MODEL),
    'checkpoint_sha256': sha256(MODEL),
    'preprocessed': str(PREP),
    'native_kernel': 'Python (stVCR benchmark)',
    'seed': SEED,
}

PRED_DIR.mkdir(parents=True, exist_ok=True)
# Write atomically so interrupted runs cannot leave a misleading .npz.
tmp_path = PRED_DIR / 'stvcr.npz.incomplete'
try:
    with open(tmp_path, 'wb') as f:
        np.savez_compressed(f, shares=shares, genes=np.array(actual_genes),
                            meta_json=np.array(json.dumps(metadata, sort_keys=True)))
    with np.load(tmp_path, allow_pickle=False) as check:
        assert check['shares'].shape == (N_PRED, 300)
        assert list(check['genes'].astype(str)) == expected_genes
        assert json.loads(str(check['meta_json']))['train_n_cells'] == 850478
    # Recheck immediately before publishing; do not replace an existing freeze.
    if PRED_FILE.exists():
        raise FileExistsError(f'Prediction appeared while running; refusing overwrite: {PRED_FILE}')
    tmp_path.rename(PRED_FILE)
finally:
    if tmp_path.exists():
        tmp_path.unlink()

print('\nSTVCR GW20 PREDICTION FROZEN:', PRED_FILE)
print('SHA-256:', sha256(PRED_FILE))
print('Native last integration time:', actual_time, '(author helper rounds times to 2 decimals)')
print('Surviving/generated population:', len(intensity))
print('Saved shape:', shares.shape, '| GW20 observations accessed: NONE')
print('Next: generate STORIES and moscot predictions, then run PHASE=score.')

## Export native EN-ET-SP endpoints for correction

In [ ]:
# No training, no loading Spatial TI/moscot/STORIES. Execute in the original
# "Python (stVCR benchmark)" kernel; outputs consumed by figure-phase notebook.
# Reproduces source selection and stVCR RK4 inference of updated 3-age figure.

from pathlib import Path
import os, sys, json, hashlib
import numpy as np
import pandas as pd
import h5py
import torch
from scipy.spatial import cKDTree

PROJECT=Path(os.environ.get('SPATIAL_TI_PROJECT',
    os.environ["STP_PROJECT"]))
DATA=Path(os.environ.get('QIAN_DATA_ROOT',
    os.environ["QIAN_PFC_DATA_ROOT"]))
STV=PROJECT/'experiments/qian_pfc_publication_4way_20261024/stvcr_slices_1_2_7_official'
OUT=PROJECT/'paper_figures/qian_ENETSP_spatial_transport_LATEST_3AGE_BARYCENTRIC_v1'
OUT.mkdir(parents=True,exist_ok=True)
STV_CSV={2:OUT/'stVCR_NEW3AGE_GW15_to_GW20_6000.csv.gz',
         7:OUT/'stVCR_NEW3AGE_GW15_to_GW22_6000.csv.gz'}
STV_RECEIPT=OUT/'stVCR_NEW3AGE_export_receipt.json'
COHORT=OUT/'GW15_ENETSP_source_cells_REDERIVED.csv.gz'
N_SV=6000
SEED=20261022
SOURCE_MATCH_RADIUS_FACTOR=0.55
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def need(p):
    p=Path(p)
    if not p.is_file():
        raise FileNotFoundError('Missing input from stVCR training: '+str(p))
    return p

def sha(p):
    h=hashlib.sha256()
    with open(need(p),'rb') as f:
        for block in iter(lambda:f.read(4<<20),b''):
            h.update(block)
    return h.hexdigest()

def decode(v):
    return v.decode('utf8') if isinstance(v,(bytes,np.bytes_)) else str(v)

def section_rows(h,sample,region):
    o=h['obs'];ss=o['sample'];rr=o['region']
    sc=[decode(v) for v in ss['categories'][:]]
    rc=[decode(v) for v in rr['categories'][:]]
    assert sample in sc and region in rc,(sample,region)
    return np.flatnonzero((ss['codes'][:]==sc.index(sample)) &
                          (rr['codes'][:]==rc.index(region)))

def h5_strings(obj,rows):
    if isinstance(obj,h5py.Group):
        cats=np.array([decode(v) for v in obj['categories'][:]],dtype=object)
        codes=np.asarray(obj['codes'][rows])
        return np.where(codes>=0,cats[np.maximum(codes,0)],'')
    return np.array([decode(v) for v in obj[rows]],dtype=object)

def original_cell_ids(h,rows):
    o=h['obs'];return h5_strings(o[decode(o.attrs.get('_index','_index'))],rows)

# Strict provenance; these MUST be the 3-age trained checkpoints.
for p in [STV/'SUCCESS.json',STV/'stvcr_preprocessed_3tissue.h5ad',
          STV/'stvcr_model_full.pt', STV/'stvcr_rigid_transform.pt',
          DATA/'gw15.h5ad',DATA/'merscope_integrated_855.h5ad']:
    need(p)
run_manifest=STV/'run_manifest.json'
if run_manifest.is_file():
    run=json.loads(run_manifest.read_text())
    assert run.get('original_slice_ids')==[1,2,7],run.get('original_slice_ids')
    assert not run.get('smoke',False),'Refusing smoke model'
success=json.loads((STV/'SUCCESS.json').read_text())
assert success.get('status')=='success',success
for key,filename in [('model_sha256','stvcr_model_full.pt'),
                     ('rigid_sha256','stvcr_rigid_transform.pt')]:
    if key in success:
        assert success[key]==sha(STV/filename),'Checkpoint changed: '+filename
print('STVCR 3-AGE INFERENCE ONLY | device:',DEVICE)
print('Checkpoint:',STV/'stvcr_model_full.pt')

# Recover EXACT source H3 cohort used by the new barycentric figure.
S={}
with h5py.File(DATA/'gw15.h5ad','r') as h:
    rows=section_rows(h,'UMB1117','F1b')
    assert len(rows)==410492,len(rows)
    xy=np.asarray(h['obsm']['spatial'][rows,:2],np.float32)
    raw_ids=original_cell_ids(h,rows).astype(str)
assert len(np.unique(raw_ids))==len(raw_ids),'Nonunique GW15 source IDs'
S[1]={'xy':xy,'raw_ids':raw_ids,
      'ids':np.array([f's01_GW15:{v}' for v in raw_ids],dtype=str)}
with h5py.File(DATA/'merscope_integrated_855.h5ad','r') as h:
    rows=section_rows(h,'UMB1117','F1b')
    hxy=np.asarray(h['obsm']['spatial'][rows,:2],np.float32)
    h3=h5_strings(h['obs']['H3_annotation'],rows)
    hh=np.fromiter((str(v).startswith('EN-ET-SP-') for v in h3),bool,count=len(h3))
    tree=cKDTree(hxy)
    subset=np.linspace(0,len(hxy)-1,min(len(hxy),2100),dtype=int)
    spacing=float(np.median(tree.query(hxy[subset],k=6)[0][:,5]))
    source_dist,source_neighbor=tree.query(xy,k=1,workers=-1)
    source_mask=(source_dist<=SOURCE_MATCH_RADIUS_FACTOR*spacing) & hh[source_neighbor]
source_rows=np.flatnonzero(source_mask)
assert len(source_rows)>=N_SV,f'Only {len(source_rows)} H3 source cells; need {N_SV}'
cohort=pd.DataFrame({'cell_id':S[1]['ids'][source_rows],
                     'stage_row':source_rows.astype(np.int64),
                     'nearest_atlas_distance':source_dist[source_rows]})
if COHORT.exists():
    old=pd.read_csv(COHORT)
    assert np.array_equal(old.cell_id.astype(str),cohort.cell_id.astype(str)), 'Source cohort drift'
    assert np.array_equal(old.stage_row.to_numpy(),cohort.stage_row.to_numpy()), 'Source cohort row drift'
else:
    cohort.to_csv(COHORT,index=False)
sv_pick=np.sort(np.random.default_rng(SEED+929).choice(source_rows,size=N_SV,replace=False))
sv_ids=S[1]['ids'][sv_pick]
print('EN-ET-SP H3 GW15 cohort:',len(source_rows),'| stVCR trajectories:',len(sv_pick))

# Run this phase in the original stVCR training environment. Zero retraining.
import scanpy as sc
for repo in [
    PROJECT/'stVCR/src', PROJECT/'stvcr/src', PROJECT/'stVCR', PROJECT/'stvcr',
    Path(str(Path(os.environ["STVCR_REPO"]) / "src")),
    Path(str(Path(os.environ["STVCR_REPO"]) / "src")),
]:
    if (repo/'stvcr'/'__init__.py').is_file():
        sys.path.insert(0,str(repo));break
import stvcr
from stvcr.training import model as _stv_model  # ensure pickle class registration
prep=sc.read_h5ad(STV/'stvcr_preprocessed_3tissue.h5ad',backed='r')
try:
    cols=set(prep.obsm)
    if not {'X_spatial_input','X_gene_input'}<=cols:raise RuntimeError('Missing stVCR official 2D/10D inputs')
    if 'original_slice_id' not in prep.obs:raise RuntimeError('No original_slice_id in stVCR preprocessing')
    pi=np.flatnonzero(prep.obs.original_slice_id.to_numpy()==1)
    if len(pi)!=410492:raise RuntimeError(f'stVCR GW15 n={len(pi)}, expected 410492')
    expected=np.array([f'1|{cid}' for cid in S[1]['raw_ids']],dtype=str)
    if not np.array_equal(prep.obs_names.to_numpy()[pi].astype(str),expected):
        raise RuntimeError('stVCR preprocessing cells differ from raw GW15 order; refusing false trajectories')
    pos=pi[sv_pick]
    xinternal=np.asarray(prep.obsm['X_spatial_input'][pos],np.float32)
    emb=np.asarray(prep.obsm['X_gene_input'][pos],np.float32)
    if emb.shape!=(N_SV,10) or xinternal.shape!=(N_SV,2):raise RuntimeError('Bad stVCR input shape')
finally:prep.file.close()
# Convert internal coordinates to physical coordinates using only the
# GW15 training source pairs. Do NOT fit any transform to heldout H3 labels.
affine=np.linalg.lstsq(np.column_stack([xinternal,np.ones(N_SV)]),S[1]['xy'][sv_pick],rcond=None)[0]
fit=np.column_stack([xinternal,np.ones(N_SV)])@affine
diameter=float(np.linalg.norm(np.ptp(S[1]['xy'],axis=0)))
res=np.linalg.norm(fit-S[1]['xy'][sv_pick],axis=1)
print('stVCR native->raw source affine max/median discrepancy:',res.max(),np.median(res),flush=True)
if np.quantile(res,.99)>0.001*diameter:
    raise RuntimeError('stVCR spatial preprocessing is not affinely convertible to source native XY: inspect alignment/scale')
net=torch.load(STV/'stvcr_model_full.pt',map_location=DEVICE,weights_only=False)
rigid=torch.load(STV/'stvcr_rigid_transform.pt',map_location=DEVICE,weights_only=False)
net.eval();rigid.eval()
if not hasattr(net,'spatial_velocity_net') or not hasattr(net,'gene_velocity_net'):
    raise RuntimeError('stVCR checkpoint has unexpected dynamics model API')
if tuple(rigid.translation.shape)!=(2,2):raise RuntimeError('Unexpected stVCR rigid transform shape')
torch.set_num_threads(min(8,torch.get_num_threads()))
xy_results={2:np.empty((N_SV,2),np.float32),7:np.empty((N_SV,2),np.float32)}
def solve_segment(y,a,b,steps):
    h=(b-a)/steps
    def vel(state,tt):
        t=torch.tensor(tt,dtype=state.dtype,device=state.device)
        return torch.cat((net.spatial_velocity_net(t,state),net.gene_velocity_net(t,state)),dim=1)
    for k in range(steps):
        t=a+k*h
        k1=vel(y,t);k2=vel(y+k1*(h/2),t+h/2)
        k3=vel(y+k2*(h/2),t+h/2);k4=vel(y+k3*h,t+h)
        y=y+(k1+2*k2+2*k3+k4)*(h/6)
    return y
with torch.inference_mode():
    for a in range(0,N_SV,512):
        b=min(N_SV,a+512)
        y=torch.as_tensor(np.column_stack([xinternal[a:b],emb[a:b]]),dtype=torch.float32,device=DEVICE)
        y=solve_segment(y,0.,5/7,184)
        out20=y[:,:2]@rigid.get_rot_matrix(0)+rigid.translation[0]
        xy_results[2][a:b]=(np.column_stack([out20.cpu().numpy(),np.ones(b-a)])@affine)
        y=solve_segment(y,5/7,1.,80)
        out22=y[:,:2]@rigid.get_rot_matrix(1)+rigid.translation[1]
        xy_results[7][a:b]=(np.column_stack([out22.cpu().numpy(),np.ones(b-a)])@affine)
        print(f'New stVCR checkpoint ODE: {b:,}/{N_SV:,}',flush=True)
for sid in (2,7):
    if not np.isfinite(xy_results[sid]).all():raise RuntimeError('Nonfinite new stVCR spatial endpoints')
rec={'run':str(STV),'model_sha256':sha(STV/'stvcr_model_full.pt'),
     'rigid_sha256':sha(STV/'stvcr_rigid_transform.pt'),
     'preprocessed_path':str(STV/'stvcr_preprocessed_3tissue.h5ad'),
     'n_source':N_SV,'source_ids_sha256':hashlib.sha256('\n'.join(sv_ids).encode()).hexdigest(),
     'source_affine':affine.tolist(), 'source_affine_p99_residual':float(np.quantile(res,.99)),
     'method':'RK4 native spatial+expression velocities 0->5/7->1; rigid map for target age',
     'growth_weighting':'not applied to individual source-cell path',
     'checkpoints_retrained':False}
if STV_RECEIPT.exists():
    prior=json.loads(STV_RECEIPT.read_text())
    if prior['model_sha256']!=rec['model_sha256'] or prior['source_ids_sha256']!=rec['source_ids_sha256']:
        raise RuntimeError('Old stVCR export uses a different checkpoint/cohort; refuse overwrite')
for sid in (2,7):
    tab=pd.DataFrame({'source_cell_id':sv_ids,'x_pred_native':xy_results[sid][:,0],
                      'y_pred_native':xy_results[sid][:,1]})
    if STV_CSV[sid].is_file():
        prior=pd.read_csv(STV_CSV[sid])
        if not np.array_equal(prior.source_cell_id.astype(str),sv_ids):
            raise RuntimeError('Saved stVCR endpoints have different source identities')
        if not np.allclose(prior[['x_pred_native','y_pred_native']],tab[['x_pred_native','y_pred_native']],atol=1e-4):
            raise RuntimeError('stVCR regeneration changed endpoints; refuse overwrite')
    else:tab.to_csv(STV_CSV[sid],index=False)
STV_RECEIPT.write_text(json.dumps(rec,indent=2))
print('STVCR EXPORT COMPLETE — now run figure phase in Spatial TI kernel.')

## Export endpoints in the raw target frame

In [ ]:
# Run ONLY in the original stVCR training environment. No retraining.
# Updates both GW15 EN-ET-SP (GW20, GW22) and H2 #3 and other frozen H2 cohorts.
# Predictions are integrated in the learned aligned COMMON frame, then converted to
# each target's physical XY by a validated, independent observation-only affine.
# Gate: If target common->raw mapping is not affine to within 0.1% tissue diameter,
# refuse publication overlays; no prediction-fitted registration or label fitting.

import os, sys, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import torch

P=Path(os.environ.get('SPATIAL_TI_PROJECT',os.environ["STP_PROJECT"]))
D=Path(os.environ.get('QIAN_DATA_ROOT',os.environ["QIAN_PFC_DATA_ROOT"]))
STV=P/'experiments/qian_pfc_publication_4way_20261024/stvcr_slices_1_2_7_official'
EN=P/'paper_figures/qian_ENETSP_spatial_transport_LATEST_3AGE_BARYCENTRIC_v1'
H2=P/'paper_figures/qian_PFC_H2_H3_Fisher_transition_matrices_TEXT_v1/H2_7_TRANSITIONS_MOSCOT_STVCR_ZOETROPE_v1'
AUD=H2/'STVCR_CORRECTED_NATIVE_EXPORT_VERIFIED_v1'; AUD.mkdir(parents=True,exist_ok=True)
PREP=STV/'stvcr_preprocessed_3tissue.h5ad'
MODEL=STV/'stvcr_model_full.pt'; RIGID=STV/'stvcr_rigid_transform.pt'
EN_COH=EN/'GW15_ENETSP_source_cells_REDERIVED.csv.gz'
EN_OLD={2:EN/'stVCR_NEW3AGE_GW15_to_GW20_6000.csv.gz',7:EN/'stVCR_NEW3AGE_GW15_to_GW22_6000.csv.gz'}
EN_NEW={2:EN/'stVCR_NEW3AGE_GW15_to_GW20_6000_UPDATED.csv.gz',7:EN/'stVCR_NEW3AGE_GW15_to_GW22_6000_UPDATED.csv.gz'}
H2_COH=H2/'FROZEN_SEVEN_H2_SOURCE_COHORT.csv.gz'
H2_NEW=H2/'STVCR_H2_FROZEN_NATIVE_SPATIAL_ENDPOINTS_UPDATED.csv.gz'
STV_RECEIPT=AUD/'stvcr_corrected_endpoint_export_receipt.json'
SPEC={1:('GW15','UMB1117','F1b',410492,0,0.),2:('GW20','FB080','F1',665204,410492,5/7),7:('GW22','FB123','F2',439986,1075696,1.)}
for f in (PREP,MODEL,RIGID,EN_COH,EN_OLD[2],EN_OLD[7],H2_COH,STV/'SUCCESS.json',STV/'run_manifest.json',STV/'training_config.json'):
    if not f.is_file():raise FileNotFoundError(f)
for f in [D/f'{SPEC[s][0].lower()}.h5ad' for s in SPEC]:
    if not f.is_file():raise FileNotFoundError(f)

def sha(f):
    h=hashlib.sha256()
    with open(f,'rb') as fh:
        for b in iter(lambda:fh.read(4<<20),b''):h.update(b)
    return h.hexdigest()

def dec(a):return a.decode() if isinstance(a,(bytes,np.bytes_)) else str(a)

suc=json.loads((STV/'SUCCESS.json').read_text())
man=json.loads((STV/'run_manifest.json').read_text())
conf=json.loads((STV/'training_config.json').read_text())
if not (suc.get('status')=='success' and man.get('smoke') is False and
        man.get('original_slice_ids')==[1,2,7] and man.get('cell_count')==1515682 and
        man.get('gene_count')==300 and man.get('autoencoder_epochs')==1000 and
        conf.get('n_epochs')==2000 and conf.get('num_samples')==500):
    raise RuntimeError('Frozen full stVCR model provenance failed')
print('PASS frozen stVCR full-run provenance',flush=True)

# Load the RAW physical section data independently, never the preprocessed obsm[spatial].
raw={}
for sid,(age,sample,region,n,_,_) in SPEC.items():
    with h5py.File(D/f'{age.lower()}.h5ad','r') as f:
        obs=f['obs']; ss=obs['sample']; rr=obs['region']
        sc=[dec(v) for v in ss['categories'][:]]
        rc=[dec(v) for v in rr['categories'][:]]
        ix=np.flatnonzero((ss['codes'][:]==sc.index(sample)) & (rr['codes'][:]==rc.index(region)))
        if len(ix)!=n:raise RuntimeError(f'RAW section {age}: expected {n}, got {len(ix)}')
        xy=np.asarray(f['obsm']['spatial'][ix,:2],np.float64)
        idxname=dec(obs.attrs.get('_index','_index'))
        ids=np.array([dec(v) for v in obs[idxname][ix]],dtype=str)
        if not np.isfinite(xy).all() or len(set(ids))!=len(ids):raise RuntimeError(f'RAW {age} invalid XY or IDs')
        raw[sid]={'xy':xy,'ids':ids,'diameter':float(np.linalg.norm(xy.max(0)-xy.min(0)))}
    print('PASS raw physical tissue',age, 'cells',n,flush=True)

# Frozen source cohorts. Exact GW15 6000 IDs from the original stVCR export.
en_old={sid:pd.read_csv(EN_OLD[sid],dtype={'source_cell_id':str}) for sid in (2,7)}
for sid in (2,7):
    if 'source_cell_id' not in en_old[sid] or len(en_old[sid])!=6000:
        raise RuntimeError(f'EN-ET-SP original GW{sid} source ID manifest invalid')
if not en_old[2]['source_cell_id'].equals(en_old[7]['source_cell_id']):
    raise RuntimeError('EN-ET-SP previous GW20/22 cohorts do not match')
en_coh=pd.read_csv(EN_COH,dtype={'cell_id':str})
if not {'cell_id','stage_row'}<=set(en_coh):raise RuntimeError('EN-ET-SP frozen cohort columns absent')
en_ids=en_old[2].source_cell_id.to_numpy(str)
lookup=en_coh.set_index('cell_id',verify_integrity=True)
en_rows=lookup.reindex(en_ids)['stage_row'].to_numpy()
if not np.isfinite(en_rows).all():raise RuntimeError('Old EN-ET-SP stVCR cell IDs not in source cohort')
en_rows=en_rows.astype(np.int64)
assert np.all((en_rows>=0)&(en_rows<len(raw[1]['ids'])))
expected_en=np.char.add('s01_GW15:',raw[1]['ids'][en_rows])
if not np.array_equal(expected_en,en_ids):raise RuntimeError('EN-ET-SP native GW15 source IDs changed')

h2=pd.read_csv(H2_COH,dtype={'source_key':str,'raw_id':str})
if not {'transition','source_sid','target_sid','source_row','source_key','raw_id'}<=set(h2):
    raise RuntimeError('H2 cohort schema mismatch')
if set(h2.transition.astype(int))!={1,2,3,4,8,9,10}:raise RuntimeError('Wrong H2 transition cohort')
for sid in (1,2):
    h=h2[h2.source_sid.eq(sid)]
    rr=h.source_row.to_numpy(np.int64)
    if not np.array_equal(h.raw_id.to_numpy(str),raw[sid]['ids'][rr]):
        raise RuntimeError(f'H2 raw cell-ID mismatch at source sid {sid}')
    if not np.array_equal(h.source_key.to_numpy(str),np.char.add(f'{sid}|',raw[sid]['ids'][rr])):
        raise RuntimeError(f'H2 stVCR cell-ID mismatch at source sid {sid}')
print('PASS exact EN-ET-SP 6000 IDs and all H2 source IDs',flush=True)

# Installed author-supplied model and saved rigid transform only.
for repo in (P/'stVCR/src',P/'stvcr/src',P/'stVCR',P/'stvcr'):
    if (repo/'stvcr/__init__.py').is_file():sys.path.insert(0,str(repo));break
import stvcr
from stvcr.training import model as _stvmodel
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net=torch.load(MODEL,map_location=DEVICE,weights_only=False)
rigid=torch.load(RIGID,map_location=DEVICE,weights_only=False)
net.eval();rigid.eval()
if tuple(rigid.translation.shape)!=(2,2):raise RuntimeError('Unexpected rigid checkpoint')
torch.set_num_threads(min(8,torch.get_num_threads()))

# Helpers are all pure coordinate transforms, not re-registrations.
def align(x,sid):
    x=np.asarray(x,np.float32)
    if sid==1:return x.astype(np.float64)
    with torch.no_grad():
        v=torch.as_tensor(x,device=DEVICE)
        i={2:0,7:1}[sid]
        y=v@rigid.get_rot_matrix(i)+rigid.translation[i]
    return y.detach().cpu().numpy().astype(np.float64)

def rows(ds,ind):
    ind=np.asarray(ind,np.int64)
    unique,inv=np.unique(ind,return_inverse=True)
    return np.asarray(ds[unique],dtype=np.float32)[inv]

def norm_residual(pred,y):return np.linalg.norm(pred-y,axis=1)

def embed_for(prep, sid, local_rows):
    off=SPEC[sid][4];global_rows=off+np.asarray(local_rows,np.int64)
    x=rows(prep['obsm']['X_spatial_input'],global_rows)
    g=rows(prep['obsm']['X_gene_input'],global_rows)
    if x.shape!=(len(local_rows),2) or g.shape!=(len(local_rows),10):raise RuntimeError('stVCR input shape drift')
    return x,g

# Fit ONLY a coordinate-system conversion from common space -> native physical XY,
# using observed-cell ID-matched coordinate pairs, with held-out QC.
B={};qc=[];TRAIN_N=3500;TEST_N=3500;SEED=20261030
with h5py.File(PREP,'r') as f:
    obs=f['obs'];name=dec(obs.attrs.get('_index','cell_key'))
    if len(obs[name])!=1515682:raise RuntimeError('Wrong preprocessed AnnData row count')
    for sid,(age,_,_,n,off,_) in SPEC.items():
        rng=np.random.default_rng(SEED+sid)
        ix=rng.choice(n,TRAIN_N+TEST_N,replace=False)
        tr=np.sort(ix[:TRAIN_N]);te=np.sort(ix[TRAIN_N:])
        # Verify randomly selected original IDs match training AnnData IDs.
        for which in (tr,te):
            trained=np.asarray(obs[name].asstr()[off+which],str)
            original=np.char.add(f'{sid}|',raw[sid]['ids'][which])
            if not np.array_equal(trained,original):
                raise RuntimeError(f'{age} original/preprocessed ID order differs: cannot fit conversion')
        xtr,_=embed_for(f,sid,tr);xte,_=embed_for(f,sid,te)
        a=align(xtr,sid);b=align(xte,sid)
        affine=np.linalg.lstsq(np.column_stack((a,np.ones(len(a)))),raw[sid]['xy'][tr],rcond=None)[0]
        p=np.column_stack((b,np.ones(len(b))))@affine
        err=norm_residual(p,raw[sid]['xy'][te]);diam=raw[sid]['diameter']
        record={'age':age,'source_train_cells':len(tr),'heldout_test_cells':len(te),
                'p99_error_raw_physical_units':float(np.quantile(err,.99)),
                'max_error_raw_physical_units':float(err.max()),
                'native_tissue_diameter':diam,'p99_fraction_of_diameter':float(np.quantile(err,.99)/diam)}
        qc.append(record)
        print(f'{age} COMMON->RAW physical inverse, heldout p99={record["p99_error_raw_physical_units"]:.6g}, '
              f'{record["p99_fraction_of_diameter"]:.4%} diameter',flush=True)
        if record['p99_fraction_of_diameter']>0.001:
            pd.DataFrame(qc).to_csv(AUD/'FAILED_RAW_COORDINATE_CONVERSION_QC.csv',index=False)
            raise RuntimeError(f'{age}: learned aligned stVCR coordinate cannot be mapped to original physical tissue by '
                               'an independently verified affine. STOP: no corrected raw-XY plots. '
                               'Use stVCR common-frame plot or inspect original preprocessing.')
        B[sid]=affine
    # Verify all requested source-cell identities from preprocessed AnnData.
    for sid,rr,checkids in ((1,en_rows,np.char.add('1|',raw[1]['ids'][en_rows])),
                            (1,h2.loc[h2.source_sid.eq(1),'source_row'].to_numpy(np.int64),
                             h2.loc[h2.source_sid.eq(1),'source_key'].to_numpy(str)),
                            (2,h2.loc[h2.source_sid.eq(2),'source_row'].to_numpy(np.int64),
                             h2.loc[h2.source_sid.eq(2),'source_key'].to_numpy(str))):
        indices=SPEC[sid][4]+rr
        uq,inv=np.unique(indices,return_inverse=True)
        found=np.asarray(obs[name].asstr()[uq],str)[inv]
        if not np.array_equal(found,checkids):raise RuntimeError('Source identities not preserved in preprocessed data')
    en_x,en_g=embed_for(f,1,en_rows)
    h2_xy=np.empty((len(h2),2),np.float32);h2_gene=np.empty((len(h2),10),np.float32)
    for sid in (1,2):
        ix=np.flatnonzero(h2.source_sid.to_numpy(int)==sid)
        rr=h2.source_row.to_numpy(np.int64)[ix]
        h2_xy[ix],h2_gene[ix]=embed_for(f,sid,rr)

# Original source colors and model time convention are fixed.
def rk4(y,a,b,steps):
    dt=(b-a)/steps
    def f(q,t):
        tt=torch.as_tensor(float(t),dtype=q.dtype,device=q.device)
        return torch.cat((net.spatial_velocity_net(tt,q),net.gene_velocity_net(tt,q)),dim=1)
    for k in range(steps):
        t=a+k*dt
        k1=f(y,t);k2=f(y+dt*k1/2,t+dt/2)
        k3=f(y+dt*k2/2,t+dt/2);k4=f(y+dt*k3,t+dt)
        y=y+dt*(k1+2*k2+2*k3+k4)/6
    return y

def final_physical(y,sid):
    common=y[:,:2].detach().cpu().numpy().astype(float)
    return np.column_stack((common,np.ones(len(common))))@B[sid]

# Prepare outputs completely in memory; write only after all QC gates pass.
# EN GW15 -> GW20 -> GW22, native continuous ODE in the common frame.
pred_en={2:np.empty((len(en_rows),2),np.float64),7:np.empty((len(en_rows),2),np.float64)}
for start in range(0,len(en_rows),512):
    end=min(start+512,len(en_rows))
    v=np.column_stack((align(en_x[start:end],1),en_g[start:end]))
    y=torch.as_tensor(v,dtype=torch.float32,device=DEVICE)
    with torch.inference_mode():
        y=rk4(y,0.,5/7,184)
        pred_en[2][start:end]=final_physical(y,2)
        y=rk4(y,5/7,1.,80)
        pred_en[7][start:end]=final_physical(y,7)
    if start%2048==0:print('EN native ODE',end,'/',len(en_rows),flush=True)

pred_h2=np.empty((len(h2),2),np.float64)
for sid,target in ((1,2),(2,7)):
    ix=np.flatnonzero(h2.source_sid.to_numpy(int)==sid)
    for start in range(0,len(ix),512):
        sub=ix[start:start+512]
        y=torch.as_tensor(np.column_stack((align(h2_xy[sub],sid),h2_gene[sub])),
                          dtype=torch.float32,device=DEVICE)
        with torch.inference_mode():
            final=rk4(y,SPEC[sid][5],SPEC[target][5],184 if sid==1 else 80)
        pred_h2[sub]=final_physical(final,target)
    print(f'H2 {SPEC[sid][0]}->{SPEC[target][0]} native ODE {len(ix):,} inputs',flush=True)
if not (all(np.isfinite(v).all() for v in pred_en.values()) and np.isfinite(pred_h2).all()):
    raise RuntimeError('Native stVCR ODE returned nonfinite physical endpoints')

# Write once; if regenerated, require same source IDs and numerically same prediction.
def write_checked(df,path,keys):
    if path.is_file():
        old=pd.read_csv(path)
        for k in keys:
            if not np.array_equal(old[k].astype(str),df[k].astype(str)):
                raise RuntimeError(f'Old corrected stVCR output uses different source identities: {path}')
        if not np.allclose(old[['x_pred_native','y_pred_native']].to_numpy(float),
                           df[['x_pred_native','y_pred_native']].to_numpy(float),atol=.03,rtol=1e-6):
            raise RuntimeError(f'Corrected stVCR output changed unexpectedly: {path}')
        print('Existing corrected output verified',path)
    else:
        df.to_csv(path,index=False)
        print('SAVED',path,flush=True)
for target in (2,7):
    df=pd.DataFrame({'source_cell_id':en_ids,'x_pred_native':pred_en[target][:,0],
                     'y_pred_native':pred_en[target][:,1]})
    write_checked(df,EN_NEW[target],['source_cell_id'])
new_h2=h2.copy();new_h2['x_pred_native']=pred_h2[:,0];new_h2['y_pred_native']=pred_h2[:,1]
write_checked(new_h2,H2_NEW,['transition','source_sid','target_sid','source_row','source_key'])
qc_table=pd.DataFrame(qc)
qc_table.to_csv(AUD/'HELDOUT_RAW_TARGET_COORDINATE_CONVERSION_QC.csv',index=False)
receipt={'checkpoint_sha256':sha(MODEL),'rigid_sha256':sha(RIGID),'preprocessing_file':str(PREP),
         'en_source_manifest_sha256':sha(EN_OLD[2]),'h2_source_manifest_sha256':sha(H2_COH),
         'cohort_n_en':len(en_rows),'cohort_n_h2':len(h2),
         'method':'source observed XY aligned by frozen rigid -> native 12D ODE -> target common XY -> independently verified target observed-coordinate inverse',
         'predictions_aligned_to_targets':False,'retrained':False,
         'target_coordinate_validation':qc,'out_en_20':str(EN_NEW[2]),'out_en_22':str(EN_NEW[7]),'out_h2':str(H2_NEW)}
if STV_RECEIPT.is_file():
    old=json.loads(STV_RECEIPT.read_text())
    if old.get('checkpoint_sha256')!=receipt['checkpoint_sha256'] or old.get('h2_source_manifest_sha256')!=receipt['h2_source_manifest_sha256']:
        raise RuntimeError('Existing corrected export receipt conflicts with these frozen checkpoints/cohorts')
else:STV_RECEIPT.write_text(json.dumps(receipt,indent=2))
print('\nPASS. Corrected native stVCR raw-target endpoints generated and verified.\n'
      'Output:',EN_NEW[2],EN_NEW[7],H2_NEW,sep='\n',flush=True)